# Solar Filament Segmentation Challenge 2026 — End-to-End Pipeline

This notebook illustrates and runs the complete two-stage instance segmentation pipeline behind our final Kaggle submission (`s2_hd_mitb2_d3_2.csv`, submission ref `58424451`, **0.39 Public / 0.37 Private LB**, **0.4598 OOF PQ** across 707 observations / 1,154 annotator views).

**Pipeline Summary**
1. **Physics-Aware Solar Disk Normalization**: Robust MAD circle fitting `(cx, cy, R)` and radial limb-darkening flat-fielding + 2D cloud leveling + local contrast normalization at both `1024x1024` and `2048x2048`.
2. **Stage-1 Heterogeneous 5-Fold Ensemble**:
   - `tile512_hrnet32`: `Unet(tu-hrnet_w32)` on overlapping `512x512` crops with late Lovasz-hinge fine-tuning (`5x5` Hanning inference at `2048px`).
   - `dfine_s_2048`: `D-FINE-seg-S` native `2048x2048` semantic decoder (`4x` D4 TTA).
   - `unet_cnx_t`: `Unet(tu-convnext_tiny)` full-disk `1024x1024` multi-task segmentation + spine head (`8x` D4 TTA).
   - `tile2048_unet_r34`: `Unet(resnet34)` `2x2` `1024px` sliding-tile segmentation (`2x` TTA).
3. **Component Tree + Hurdle LightGBM + Antichain DP (`dp_hd_sharp`)**: Converts multi-model probability maps into a 15-level Component Tree (`tau = 0.14..0.80`), scores expected PQ utility `P(hit) * E[IoU | hit]` using cross-fitted Hurdle LightGBM, and selects optimal disjoint instances via dynamic programming (`tau = 0.24`, `w_2048 = 0.92`).
4. **Stage-2 Canonical Rotated Crop Refiner (`s2_hd_mitb2`)**: Warps each candidate instance into a principal-axis-aligned `128x384` 6-channel crop and refines boundaries + instance existence score (`comb = 0.65 * keep + 0.35 * s1_score >= 0.48`, `tau = 0.50`, `min_area = 120`, `top_k = 50`).

Set `RETRAIN_FROM_SCRATCH = False` (default) to download Release `v1.0` checkpoints and reproduce `submission.csv` directly from raw test images, or `True` to train all models from scratch first.

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from pycocotools import mask as mask_utils

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts import download_weights
from src import cache, cache_2048, data, export, geometry, metrics, paths

# Configure dataset & output paths
DATA_ROOT = Path(os.environ.get("SOLAR_DATA_ROOT", REPO_ROOT / "data" / "MAGFiLO_1.0_Kaggle_2026"))
RUNS_DIR = Path(os.environ.get("SOLAR_RUNS_DIR", REPO_ROOT / "runs"))
OUT_CSV = REPO_ROOT / "submission.csv"

RETRAIN_FROM_SCRATCH = False  # Set True only if retraining all 5-fold Stage-1 + Stage-2 models
WORKERS = max(1, min(24, (os.cpu_count() or 8) - 4))
DFINE_PYTHON = os.environ.get("SOLAR_DFINE_PYTHON", sys.executable)

os.environ["SOLAR_REPO_ROOT"] = str(REPO_ROOT)
os.environ["SOLAR_DATA_ROOT"] = str(DATA_ROOT)
os.environ["SOLAR_RUNS_DIR"] = str(RUNS_DIR)

print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"Workers: {WORKERS} | DATA_ROOT: {DATA_ROOT} | RUNS_DIR: {RUNS_DIR}")

## 1. Download Verified Checkpoints (Release v1.0)

When reproducing from pre-trained weights (`RETRAIN_FROM_SCRATCH = False`), `download_weights.ensure_weights()` fetches and SHA-256 verifies all 5-fold Stage-1 models, the fitted `dp_hd_sharp` Hurdle LightGBM bundle, and the 5-fold Stage-2 `Unet(mit_b2)` refiner weights into `runs/`.

In [ ]:
if not RETRAIN_FROM_SCRATCH:
    download_weights.ensure_weights(runs_dir=RUNS_DIR)
else:
    print("RETRAIN_FROM_SCRATCH=True: skipping checkpoint download.")

## 2. Physics Normalization & Feature Caching

For every `2048x2048` H-alpha frame we fit the solar disk circle `(cx, cy, R)` via robust MAD outlier rejection (`src/geometry.py`) and construct 3 channels (`disk_z`, radial + 2D flat-field, local contrast z-score) at `1024x1024` and `2048x2048`. Below we run caching and visualize one test observation before and after normalization.

In [ ]:
env = os.environ.copy()

def run_cmd(cmd):
    print(">>>", " ".join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=str(REPO_ROOT), env=env)

test_flag = [] if RETRAIN_FROM_SCRATCH else ["--test-only"]
run_cmd([sys.executable, "-u", "-m", "src.cache", *test_flag, "--workers", str(WORKERS)])
run_cmd([sys.executable, "-u", "-m", "src.cache_2048", *test_flag, "--workers", str(WORKERS)])

test_imgs = sorted((DATA_ROOT / "test" / "test_images").glob("*.jpeg"))
sample_stem = test_imgs[0].stem
raw_gray = cv2.imread(str(test_imgs[0]), cv2.IMREAD_GRAYSCALE)
norm_hwc = cv2.imread(str(paths.CACHE_2048 / "img" / f"{sample_stem}.png"), cv2.IMREAD_COLOR)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), dpi=130)
titles = ["Raw GONG H-alpha", "Ch0: Disk Z-Score", "Ch1: Flat-Field", "Ch2: Local Contrast"]
imgs = [raw_gray, norm_hwc[:, :, 0], norm_hwc[:, :, 1], norm_hwc[:, :, 2]]
for ax, title, im in zip(axes, titles, imgs):
    ax.imshow(cv2.resize(im, (512, 512), interpolation=cv2.INTER_AREA), cmap="gray")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
fig.suptitle(f"Solar Disk Physics Normalization Example ({sample_stem})", fontsize=11)
plt.tight_layout()
plt.show()

## 3. Optional Full Training from Scratch

When `RETRAIN_FROM_SCRATCH = True`, all 5 grouped folds (`StratifiedGroupKFold` by `station x annotator multiplicity`) of the four Stage-1 models and the Tree-DP + Stage-2 refiner are trained end-to-end. By default (`False`), execution moves straight to inference on the loaded checkpoints.

In [ ]:
if RETRAIN_FROM_SCRATCH:
    # Stage-1 Model A: 1024px ConvNeXt-Tiny U-Net
    run_cmd([
        sys.executable, "-u", "-m", "src.train_seg",
        "--arch", "Unet", "--encoder", "tu-convnext_tiny",
        "--run-name", "unet_cnx_t", "--folds", "0,1,2,3,4",
        "--epochs", "24", "--batch-size", "4", "--lr", "3e-4",
        "--n-tta", "8", "--channels-last",
    ])
    # Stage-1 Model B: 1024px 2x2 Tiled ResNet-34 U-Net
    run_cmd([
        sys.executable, "-u", "-m", "src.train_tile2048",
        "--arch", "Unet", "--encoder", "resnet34",
        "--run-name", "tile2048_unet_r34", "--folds", "0,1,2,3,4",
        "--epochs", "24", "--batch-size", "4", "--lr", "4e-4", "--n-tta", "2",
    ])
    # Stage-1 Model C: 512px Overlapping-Tile HRNet-W32 + Lovasz
    run_cmd([
        sys.executable, "-u", "-m", "src.train_tile512",
        "--arch", "Unet", "--encoder", "tu-hrnet_w32",
        "--run-name", "tile512_hrnet32", "--folds", "0,1,2,3,4",
        "--epochs", "20", "--crop-size", "512", "--overlap", "0.25",
        "--pos-weight", "2.0", "--lovasz-weight", "0.35", "--n-tta", "4", "--no-arch",
    ])
    # Stage-1 Model D: Native 2048px D-FINE-seg-S
    run_cmd([
        DFINE_PYTHON, "-u", "-m", "src.train_dfine",
        "--mode", "train", "--run-name", "dfine_s_2048",
        "--folds", "0,1,2,3,4", "--epochs", "30", "--tta", "4",
    ])
    # Fit Tree-DP + Hurdle LightGBM aggregator
    run_cmd([
        sys.executable, "-u", "-m", "src.run_tree_dp",
        "--runs", "tile512_hrnet32,dfine_s_2048,unet_cnx_t,tile2048_unet_r34",
        "--weights", "0.38,0.34,0.16,0.12",
        "--tile2048-run", "tile512_hrnet32,dfine_s_2048,tile2048_unet_r34",
        "--tile2048-weights", "0.46,0.40,0.14",
        "--w-2048", "0.92", "--weighted-records",
        "--workers", str(WORKERS), "--export-sub", "--out-name", "dp_hd_sharp",
    ])
    # Train Stage-2 canonical rotated crop refiner
    run_cmd([
        sys.executable, "-u", "-m", "src.stage2_refine",
        "--dp-name", "dp_hd_sharp", "--out-name", "s2_hd_mitb2",
        "--arch", "Unet", "--encoder", "mit_b2", "--folds", "0,1,2,3,4",
        "--epochs", "16", "--batch-size", "24", "--lr", "4e-4", "--n-tta", "4",
        "--workers", str(WORKERS),
    ])
else:
    print("Using pre-trained Release v1.0 checkpoints.")

## 4. Test-Set Inference: Stage-1 Ensemble -> Tree DP -> Stage-2 Refiner

Next we run all 5 folds of each Stage-1 model on the 180 test images, score the multi-threshold Component Tree with the Hurdle LightGBM models (`dp_hd_sharp`, `tau = 0.24`), and run the 5-fold Stage-2 `Unet(mit_b2)` refiner on canonical `128x384` rotated crops (`comb = 0.65 * keep + 0.35 * s1_score >= 0.48`, `tau = 0.50`).

In [ ]:
if not RETRAIN_FROM_SCRATCH:
    run_cmd([
        sys.executable, "-u", "-m", "src.train_seg",
        "--arch", "Unet", "--encoder", "tu-convnext_tiny",
        "--run-name", "unet_cnx_t", "--folds", "0",
        "--n-tta", "8", "--channels-last", "--infer-test",
    ])
    run_cmd([
        sys.executable, "-u", "-m", "src.train_seg",
        "--arch", "Unet", "--encoder", "tu-convnext_tiny",
        "--run-name", "unet_cnx_t", "--folds", "1,2,3,4",
        "--n-tta", "4", "--channels-last", "--infer-test",
    ])
    run_cmd([
        sys.executable, "-u", "-m", "src.train_tile2048",
        "--arch", "Unet", "--encoder", "resnet34",
        "--run-name", "tile2048_unet_r34", "--folds", "0,1,2,3,4",
        "--n-tta", "2", "--infer-test",
    ])
    run_cmd([
        sys.executable, "-u", "-m", "src.train_tile512",
        "--arch", "Unet", "--encoder", "tu-hrnet_w32",
        "--run-name", "tile512_hrnet32", "--folds", "0,1,2,3,4",
        "--crop-size", "512", "--zoom", "1", "--overlap", "0.25",
        "--pos-weight", "2.0", "--n-tta", "4", "--channels-last", "--infer-test",
    ])
    run_cmd([
        DFINE_PYTHON, "-u", "-m", "src.train_dfine",
        "--mode", "infer-test", "--run-name", "dfine_s_2048",
        "--folds", "0,1,2,3,4", "--tta", "4",
    ])
    run_cmd([
        sys.executable, "-u", "-m", "src.run_tree_dp",
        "--test-only", "--out-name", "dp_hd_sharp",
        "--bundle", str(RUNS_DIR / "dp_hd_sharp" / "tree_dp_bundle.pkl"),
        "--workers", str(WORKERS), "--min-area-2048", "120",
    ])

run_cmd([
    sys.executable, "-u", "-m", "src.stage2_refine",
    "--test-only", "--dp-name", "dp_hd_sharp", "--out-name", "s2_hd_mitb2",
    "--arch", "Unet", "--encoder", "mit_b2",
    "--batch-size", "24", "--n-tta", "4",
    "--alpha", "1.0", "--tau", "0.50", "--comb-thr", "0.48",
    "--min-area-2048", "120", "--workers", str(WORKERS),
    "--out-csv", str(OUT_CSV),
])

## 5. Submission Verification & Visual Inspection

Finally, we verify schema, non-emptiness across all 180 test images, disjointness & RLE round-trip fidelity on `submission.csv`, and plot a predicted instance overlay on a sample test observation.

In [ ]:
sub_df = pd.read_csv(OUT_CSV)
print(f"Submission path: {OUT_CSV}")
print(f"Total rows (predicted filament instances): {len(sub_df)}")
stems_present = sub_df["filament_id"].str.rsplit("_", n=1).str[0]
counts = stems_present.value_counts()
print(f"Images represented: {len(counts)} / {len(test_imgs)} | instances/img min={counts.min()} med={counts.median():.1f} max={counts.max()}")

# Verify zero overlaps and visualize first sample stem predictions
sample_rows = sub_df[stems_present == sample_stem]
base_rgb = np.stack([norm_hwc[:, :, 1]] * 3, axis=-1).astype(np.float32)
overlay = base_rgb.copy()
for rle_s in sample_rows["segmentation_rle"]:
    m = mask_utils.decode(metrics.from_counts(rle_s, 2048, 2048)) > 0
    overlay[m] = 0.65 * overlay[m] + 0.35 * np.array([180.0, 95.0, 255.0])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5), dpi=130)
ax1.imshow(cv2.resize(norm_hwc[:, :, 1], (768, 768)), cmap="gray")
ax1.set_title(f"{sample_stem}: Normalized Flat-Field", fontsize=10)
ax1.axis("off")
ax2.imshow(cv2.resize(np.clip(overlay, 0, 255).astype(np.uint8), (768, 768)))
ax2.set_title(f"{sample_stem}: Predicted Instances ({len(sample_rows)})", fontsize=10)
ax2.axis("off")
plt.tight_layout()
plt.show()